# Lara's notities

## Preprocessen/sample methoden
The pipeline described by MNE is https://mne.tools/stable/documentation/cookbook.html is VERY detailed. I think our data is already quite polished maybe? They only talk about downsampling and normalization in the project description so maybe such things are more complicated than needed. 
Do downsampling with basic factor 10 right now and do average pooling, factor can be bigger when slow. PCA for dimension reduction?
Z-score normalization can be better then min-max when amplitudes vary a lot for the MEG data (z-score uses the normal distribution while min-max is linear, https://www.codecademy.com/article/min-max-zscore-normalization).

## (a) Modellen om uit te kiezen
MNE tools are made for MEG analysis and visualization. Also learning? sklearn. Could do normal classification too maybe. I chose 1D CNN becauseee it seems fine (survey: https://www.sciencedirect.com/science/article/pii/S0888327020307846).
optimizer adam sure. do entropyloss as a loss function? We do memory management with batches of size 16.
I have not done cross validation on parameters yet. En moet dus nog een model toevoegen.

## (b) Test framework voor zowel cross als intra testen
ja gedaan. Geen cross validation.

## (c) Hyperparameter tweaking framework
TO DO!

We moeten vooral ook beschrijven hoe we verbeteren enzo. Miss een soort labjournaal bijhouden?

In [ ]:
%pip install torch numpy scipy scikit-learn matplotlib h5py

# Read data

In [ ]:
import os
import glob
import h5py
import numpy as np

from sklearn.

# Functions to load/get/extract adminstrative stuff
def get_dataset_name(file_name_with_dir):
    file_name_without_dir = file_name_with_dir.split("/")[-1]
    temp = file_name_without_dir.split(".")[:-1]
    dataset_name = "".join(temp)
    return dataset_name

def load_h5_file(filepath):
    f = h5py.File(filepath, 'r')
    dataset_name = get_dataset_name(filepath)
    return f.get(dataset_name)[()]      # Outputs matrix that is the numpy array of the data in the h5py file

def get_label(filepath):    # extract classification activity label: 0="rest", 1="math", 2="memory" en 3="motor"
    name = os.path.basename(filepath)
    if name.startswith("rest"):     # maybe slow sorry, idk how else to do this
        return 0
    elif name.startswith("math"):
        return 1
    elif name.startswith("memory"):
        return 2
    elif name.startswith("motor"):
        return 3

# downsample when loading data
def downsample(matrix, factor=10):
    nsensors, ntimesteps = matrix.shape
    down_ntimesteps = ntimesteps // factor
    matrix = matrix[:, :down_ntimesteps * factor]
    matrix = matrix.reshape(nsensors,down_ntimesteps,factor).mean(axis=2)
    return matrix

# extract useful properties
def extract_properties(matrix):
    mean = np.mean(matrix, axis=1)
    std = np.std(matrix, axis=1)
    max_value = np.max(matrix, axis=1)
    min_value = np.min(matrix, axis=1)
    return np.concatenate([mean,std,min_value,max_value])

# load a folder of data!
def load_folder(folder):
    x = []
    y = []
    files = glob.glob("data/" + folder + "/*.h5")
    for file in files:
        matrix = load_h5_file(file)
        matrix = downsample(matrix)
        props = extract_properties(matrix)
        x.append(props)
        y.append(get_label(file))
    return np.array(x), np.array(y)

# Load all the data (maybe requires too much memory?)
x_train_intra, y_train_intra = load_folder("Intra/train")
x_test_intra, y_test_intra = load_folder("Intra/test")

x_train_cross, y_train_cross = load_folder("Cross/train")
x_test_cross1, y_test_cross1 = load_folder("Cross/test1")
x_test_cross2, y_test_cross2 = load_folder("Cross/test2")
x_test_cross3, y_test_cross3 = load_folder("Cross/test3")

# Preprocessing
z-score normalization

In [ ]:
from sklearn.preprocessing import StandardScaler

# Normalization via z-score sklearn standardscaler
def normalize(train, test):
    # scale train
    n, c, t = train.shape
    train = train.reshape(n, -1)    # reshape to apply standardscaler (sorry if ugly)
    scaler = StandardScaler()
    train = scaler.fit_transform(train)
    train = train.reshape(n,c,t)    # reshape back

    # scale test
    ntest = test.shape[0]
    test = test.reshape(ntest, -1)    # reshape to apply standardscaler (sorry if ugly)
    test = scaler.transform(test)
    test = test.reshape(ntest,c,t)    # reshape back
    return train, test

# Normalize all data (repeated code oh well)
x_train_intra, x_test_intra = normalize(x_train_intra, x_test_intra)
x_train_cross, x_test_cross1 = normalize(x_train_cross, x_test_cross1)
_, x_test_cross2 = normalize(x_train_cross, x_test_cross2)
_, x_test_cross3 = normalize(x_train_cross, x_test_cross3)

# Model training and testing methods

In [ ]:
import torch
from torch.utils.data import Dataset
import torch.nn as nn

from torch.utils.data import DataLoader
import torch.optim as optim

# We use torch once again
class Torchdata(Dataset):
    def __init__(self,x,y):
        self.x = torch.tensor(x,dtype=float32)
        self.y = torch.tensor(y,dtype=torch.long)
    
    def __len__(self):
        return len(self.y)
    
    def __getitem__(self,xid):
        return self.x[xid], self.y[xid]

# we use the 1d convolutional nn
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv1d(248, 64, kernel_size=7, padding=3)
        self.conv2 = nn.Conv1d(64, 128, kernel_size=5, padding=2)
        self.pool = nn.MaxPool1d(4)
        self.relu = nn.ReLU()       # ReLu in action! im pfff i need a small break.
        self.fc1 = nn.Linear(128 * 445, 128)
        self.fc2 = nn.Linear(128, 4)

    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.pool(x)
        x = self.relu(self.conv2(x))
        x = self.pool(x)
        x = torch.flatten(x, 1)
        x = self.relu(self.fc1(x))
        return self.fc2(x)

# Trainng method. Epochs can change ofc. Not yet cross validation on parameters
def train_model(x_train,y_train,epochs=100):
    dataset = Torchdata(x_train,y_train)
    loader = DataLoader(dataset, batch_size=16, shuffle=True)       # memory management!
    model = CNN()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    optimizer = optim.Adam(model.parameters(),lr=0.001)
    loss_function = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = loss_function(outputs, y_batch)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print("Epoch", epoch, "Loss:", total_loss)
    return model
    
# Evaluation, returns accuracy
def evaluate(model, x, y):
    dataset = Torchdata(x, y)
    loader = DataLoader(dataset, batch_size=16)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.eval()

    correct = 0
    total = 0
    with torch.no_grad():
        for x_batch, y_batch in loader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)
            outputs = model(x_batch)
            _, predicted = torch.max(outputs, 1)
            total += y_batch.size(0)
            correct += (predicted == y_batch).sum().item()
    return correct / total

# Do the training and the testing

In [ ]:
# Intra
model_intra = train_model(x_train_intra, y_train_intra)
acc_intra = evaluate(model_intra, x_test_intra, y_test_intra)
print("Intra-subject accuracy:", acc_intra)

# Cross
model_cross = train_model(x_train_cross, y_train_cross)
acc_cross1 = evaluate(model_cross, x_test_cross1, y_test_cross1)
acc_cross2 = evaluate(model_cross, x_test_cross2, y_test_cross2)
acc_cross3 = evaluate(model_cross, x_test_cross3, y_test_cross3)
print("Cross-subject 1 accuracy:", acc_cross1)
print("Cross-subject 2 accuracy:", acc_cross2)
print("Cross-subject 3 accuracy:", acc_cross3)